# EXP-005 — Tách head ảnh và head text (seed 1)
Train từ cache như EXP-004 (`crossA`, λ = 1, 30 epoch). Khác biệt: text đi qua **head riêng W_t** (khởi tạo = W_i), học bằng loss neo. Head ảnh W_i học InfoNCE.
- `sg`: loss neo chỉ cập nhật W_t → W_i giống hệt head λ = 0 (EXP-003 seed 1).
- `joint`: loss neo cập nhật cả W_i và W_t.
- `μ`: phạt `μ‖W_t − W_i‖²` để W_t không lệch quá xa (W_t chỉ thấy 336 câu).

Quét `sg`/`joint` × μ ∈ {0, 0.1}, test ở epoch 5 / 10 / 20 / 30, so với EXP-004 λ = 1 (head chung), λ = 0 (EXP-003) và EXP-000 seed1.

Mục tiêu: RAFDB Ensemble Five cao hơn EXP-004 λ = 1, các dataset khác không thua quá 0.5.

# Cấu hình

In [3]:
SEED = 1
EPOCHS = 30
SAVE_EPOCHS = [5, 10, 20, 30]         # test head ở các mốc này
LR = 1e-3
LAMBDA = 1                            # trọng số loss neo (chốt từ EXP-004)
CONFIGS = [("sg", 0), ("sg", 0.1), ("joint", 0), ("joint", 0.1)]   # (sep-head, μ)
job_of = lambda mode, mu: f"exp005_sephead_{mode}_mu{mu:g}_l14_seed{SEED}"
BRANCH = "exp/005-separate-text-head"

INSTRUCTION = "Please play the role of a facial action describer. Objectively describe the detailed facial actions of the person in the image."
TRAIN_DATA = "/content/CAER-S_Face/train"
FACE_ZIP = "/content/drive/MyDrive/Datasets/CAERS_Face_train.zip"
TEST_ZIP = "/content/drive/MyDrive/Datasets/static_test_sets.zip"
PRE_DIR = "/content/drive/MyDrive/ExpCLIP_output/exp001/full"     # target đã cache ở EXP-001 (targets/chunk_*.npz)
CLIP_CACHE = f"{PRE_DIR}/clip_feats_l14.npz"
EXP004_DIR = "/content/drive/MyDrive/ExpCLIP_output/exp004"
BANK = f"{EXP004_DIR}/text_bank_A_l14.npz"                         # ngân hàng A của EXP-004
OUT_DIR = "/content/drive/MyDrive/ExpCLIP_output/exp005"
EXP004_CSV = f"{EXP004_DIR}/exp004_anchor_crossA_l14_seed1-results.csv"                      # λ = 1 (head chung)
EXP003_CSV = "/content/drive/MyDrive/ExpCLIP_output/exp003/exp003_cache_l14_seed1-results.csv"   # λ = 0
EXP003_DIR = "/content/drive/MyDrive/ExpCLIP_output/exp003"

import os, subprocess
def run(cmd):
    """Chạy lệnh shell, in output trực tiếp; lỗi thì dừng cell (khác với `!`)."""
    # bash + pipefail: lệnh có `| tee` vẫn báo lỗi nếu python lỗi
    p = subprocess.Popen("set -o pipefail; " + cmd, shell=True, executable="/bin/bash", text=True,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         env={**os.environ, "PYTHONUNBUFFERED": "1"})  # in log ngay, không đợi hết
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"Lệnh lỗi (exit {p.returncode}) — xem output ở trên")

# Mount Drive & GPU

In [4]:
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(OUT_DIR, exist_ok=True)
# không nạp BLIP-2 → GPU nào cũng được (T4/L4/A100)
!nvidia-smi --query-gpu=name,memory.total --format=csv

Mounted at /content/drive
name, memory.total [MiB]
Tesla T4, 15360 MiB


# Clone repo (nhánh EXP-005)

In [5]:
if not os.path.isdir("/content/Exp-CLIP"):
    !git clone -b $BRANCH https://github.com/huongng389/Exp-CLIP.git /content/Exp-CLIP
%cd /content/Exp-CLIP
assert os.getcwd() == "/content/Exp-CLIP", "clone lỗi — nhánh đã push chưa?"
!git checkout $BRANCH && git pull
!git log --oneline -1   # ghi lại commit vào research_log

Cloning into '/content/Exp-CLIP'...
remote: Enumerating objects: 419, done.
remote: Counting objects: 100% (419/419), done.
remote: Compressing objects: 100% (148/148), done.
remote: Total 419 (delta 307), reused 378 (delta 267), pack-reused 0 (from 0)
Receiving objects: 100% (419/419), 32.25 MiB | 32.83 MiB/s, done.
Resolving deltas: 100% (307/307), done.
/content/Exp-CLIP
Already on 'exp/005-separate-text-head'
Your branch is up to date with 'origin/exp/005-separate-text-head'.
Already up to date.
e1f45ec (HEAD -> exp/005-separate-text-head, origin/exp/005-separate-text-head) EXP-005: separate text head trained by the anchor loss


# Giải nén dữ liệu (train + test)

In [6]:
!mkdir -p /content/CAER-S_Face /content/static_test_sets
!unzip -q -n "$FACE_ZIP" -d /content/CAER-S_Face
!unzip -q -n "$TEST_ZIP" -d /content/static_test_sets
!ls /content/CAER-S_Face/train
!ls /content/static_test_sets

Angry  Disgust	Fear  Happy  Neutral  Sad  Surprise
AffectNet7  AffectNet8	FERPlus  RAFDB


# Cài dependencies (venv310)\ntorch 2.0.1 + lavis: dùng cho target, proto, weight, train, test.

In [7]:
!sudo apt-get -qq update > /dev/null
!sudo apt-get -qq install -y python3.10 python3.10-venv python3.10-distutils > /dev/null
!python3.10 -m venv /content/venv310
!/content/venv310/bin/pip install -q --upgrade pip
!/content/venv310/bin/pip install -q --no-cache-dir torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118
!/content/venv310/bin/pip install -q --no-cache-dir transformers tqdm regex pyyaml pandas pydantic click timm einops
!/content/venv310/bin/pip install -q --no-cache-dir salesforce-lavis
# phải cài SAU lavis: lavis kéo numpy 2.x / opencv mới
!/content/venv310/bin/pip install -q --no-cache-dir "numpy<2" "opencv-python==4.10.0.84" "opencv-python-headless==4.10.0.84"
!/content/venv310/bin/pip install -q --no-cache-dir matplotlib scikit-learn
!/content/venv310/bin/python -c "import numpy, torch, lavis, sklearn; print('numpy', numpy.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())"

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 7.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 16.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Instal

# Kiểm tra cache
Feature ảnh (EXP-003) và ngân hàng A (EXP-004) đã có trên Drive.

In [8]:
assert os.path.exists(CLIP_CACHE) and os.path.exists(BANK), "thiếu cache — chạy notebook EXP-004 trước"
os.makedirs(OUT_DIR, exist_ok=True)
!ls -lh "$CLIP_CACHE" "$BANK"

-rw------- 1 root root  75M Oct 14 07:10 /content/drive/MyDrive/ExpCLIP_output/exp001/full/clip_feats_l14.npz
-rw------- 1 root root 576K Oct 14 15:16 /content/drive/MyDrive/ExpCLIP_output/exp004/text_bank_A_l14.npz


# Train từng cấu hình (30 epoch, từ cache)
Bị ngắt thì chạy lại cell này: cấu hình đã xong được bỏ qua, cấu hình đang dở tự resume.

In [9]:
for mode, mu in CONFIGS:
    job = job_of(mode, mu)
    if os.path.exists(f"{OUT_DIR}/{job}-e{SAVE_EPOCHS[-1]}-text-model.pth"):
        print(f"{mode} μ={mu}: đã train xong, bỏ qua"); continue
    resume = f"{OUT_DIR}/{job}-resume.pth"
    print(f"=============== {mode}, μ = {mu}")
    run(f'/content/venv310/bin/python train.py '
        f'--epochs {EPOCHS} --batch-size 512 --batch-size-test-image 512 --batch-size-test-video 64 '
        f'--lr {LR} --weight-decay 1e-4 --momentum 0.9 --print-freq 1000 --milestones 1000 --seed {SEED} '
        f'--job-id {job} --instruction "{INSTRUCTION}" --load-model CLIP_L14 '
        f'--cache-clip {CLIP_CACHE} --cache-targets {PRE_DIR}/targets '
        f'--text-bank {BANK} --anchor-weight {LAMBDA} --anchor-type cross '
        f'--sep-head {mode} --head-tie {mu} '
        f'--save-epochs {" ".join(map(str, SAVE_EPOCHS))} --extra-checkpoint-dir {OUT_DIR}'
        + (f' --resume {resume}' if os.path.exists(resume) else ''))

=============== sg, μ = 0
Training date:  26-10-18 02:52
************************
workers = 8
epochs = 30
batch_size = 512
batch_size_test_image = 512
batch_size_test_video = 64
lr = 0.001
weight_decay = 0.0001
momentum = 0.9
print_freq = 1000
milestones = [1000]
seed = 1
job_id = exp005_sephead_sg_mu0_l14_seed1
instruction = Please play the role of a facial action describer. Objectively describe the detailed facial actions of the person in the image.
load_model = CLIP_L14
extra_checkpoint_dir = /content/drive/MyDrive/ExpCLIP_output/exp005
train_data = /data/EECS-IoannisLab/datasets/Static_FER_Datasets/CAERS_Face/train
sample_weights = None
cache_clip = /content/drive/MyDrive/ExpCLIP_output/exp001/full/clip_feats_l14.npz
cache_targets = /content/drive/MyDrive/ExpCLIP_output/exp001/full/targets
text_bank = /content/drive/MyDrive/ExpCLIP_output/exp004/text_bank_A_l14.npz
anchor_weight = 1.0
anchor_type = cross
sep_head = sg
head_tie = 0.0
save_epochs = [5, 10, 20, 30]
resume = None
*****

# Kiểm tra: `sg` có head ảnh giống λ = 0 không
Với `sg`, W_i chỉ học InfoNCE nên phải trùng (gần đúng, do GPU không tất định) với head EXP-003 seed 1 ở cùng epoch.

In [10]:
import torch
for e in SAVE_EPOCHS:
    ref = f"{EXP003_DIR}/exp003_cache_l14_seed1-e{e}-model.pth"
    if not os.path.exists(ref):
        print("không thấy", ref); continue
    a = torch.load(ref, map_location="cpu")["mlp.weight"]
    b = torch.load(f"{OUT_DIR}/{job_of('sg', 0)}-e{e}-model.pth", map_location="cpu")["mlp.weight"]
    print(f"epoch {e}: ‖W_i(sg) − W(λ=0)‖ / ‖W(λ=0)‖ = {((a - b).norm() / a.norm()).item():.2e}")

epoch 5: ‖W_i(sg) − W(λ=0)‖ / ‖W(λ=0)‖ = 4.53e-08
epoch 10: ‖W_i(sg) − W(λ=0)‖ / ‖W(λ=0)‖ = 6.73e-08
epoch 20: ‖W_i(sg) − W(λ=0)‖ / ‖W(λ=0)‖ = 9.54e-08
epoch 30: ‖W_i(sg) − W(λ=0)‖ / ‖W(λ=0)‖ = 1.16e-07


# Test zero-shot từng mốc
`test.py` tự nạp `<job>-text-model.pth` nếu có.

In [11]:
!mkdir -p checkpoint log
for mode, mu in CONFIGS:
    for e in SAVE_EPOCHS:
        job = f"{job_of(mode, mu)}-e{e}"
        if os.path.exists(f"{OUT_DIR}/{job}-test.txt"):
            !cp "$OUT_DIR/$job-test.txt" log/
            continue
        !cp "$OUT_DIR/$job-model.pth" "$OUT_DIR/$job-text-model.pth" checkpoint/
        print(f"=============== {mode}, μ = {mu}, epoch {e}")
        run(f'/content/venv310/bin/python test.py --load-model CLIP_L14 --job-id {job} '
            f'--data-root /content/static_test_sets 2>&1 | tee log/{job}-test.txt')
        !grep -q "Separate text head loaded" log/$job-test.txt || echo "CẢNH BÁO: không nạp head text"
        !cp log/$job-test.txt "$OUT_DIR/"

=============== sg, μ = 0, epoch 5
************************
load_model = CLIP_L14
job_id = exp005_sephead_sg_mu0_l14_seed1-e5
data_root = /content/static_test_sets
************************
Separate text head loaded from ./checkpoint/exp005_sephead_sg_mu0_l14_seed1-e5-text-model.pth
************************************************************************** Zero-shot Prompt Type:  Class Name
******************** Static FER Zero-shot Performance ********************
************************* RAFDB
UAR/WAR: 49.87/51.86
************************* AffectNet7
UAR/WAR: 41.11/41.11
************************* AffectNet8
UAR/WAR: 35.67/35.67
************************* FERPlus
UAR/WAR: 43.01/44.72
************************************************************************** Zero-shot Prompt Type:  An Expression of Name
******************** Static FER Zero-shot Performance ********************
************************* RAFDB
UAR/WAR: 58.68/62.45
************************* AffectNet7
UAR/WAR: 45.34/45.34
*

# So sánh
Bảng 1: Ensemble Five (UAR) theo cấu hình × epoch, kèm EXP-004 λ = 1 (head chung) và λ = 0. Bảng 2: Δ so với EXP-004 λ = 1 cùng epoch. Bảng 3: đủ 5 prompt ở epoch 30, Δ so với EXP-004 λ = 1.

In [12]:
import re, pandas as pd
REF = pd.DataFrame([{"prompt": "Class Name", "dataset": "RAFDB", "src": "EXP-000 seed1", "UAR": 50.36, "WAR": 56.58}, {"prompt": "Class Name", "dataset": "AffectNet7", "src": "EXP-000 seed1", "UAR": 42.51, "WAR": 42.51}, {"prompt": "Class Name", "dataset": "AffectNet8", "src": "EXP-000 seed1", "UAR": 36.95, "WAR": 36.95}, {"prompt": "Class Name", "dataset": "FERPlus", "src": "EXP-000 seed1", "UAR": 38.27, "WAR": 45.45}, {"prompt": "An Expression of Name", "dataset": "RAFDB", "src": "EXP-000 seed1", "UAR": 56.73, "WAR": 64.05}, {"prompt": "An Expression of Name", "dataset": "AffectNet7", "src": "EXP-000 seed1", "UAR": 44.0, "WAR": 44.0}, {"prompt": "An Expression of Name", "dataset": "AffectNet8", "src": "EXP-000 seed1", "UAR": 38.33, "WAR": 38.33}, {"prompt": "An Expression of Name", "dataset": "FERPlus", "src": "EXP-000 seed1", "UAR": 39.99, "WAR": 52.92}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "RAFDB", "src": "EXP-000 seed1", "UAR": 54.94, "WAR": 65.12}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "AffectNet7", "src": "EXP-000 seed1", "UAR": 43.97, "WAR": 43.97}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "AffectNet8", "src": "EXP-000 seed1", "UAR": 38.02, "WAR": 38.02}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "FERPlus", "src": "EXP-000 seed1", "UAR": 36.63, "WAR": 48.89}, {"prompt": "Expression Ensemble Five", "dataset": "RAFDB", "src": "EXP-000 seed1", "UAR": 58.88, "WAR": 66.88}, {"prompt": "Expression Ensemble Five", "dataset": "AffectNet7", "src": "EXP-000 seed1", "UAR": 44.09, "WAR": 44.09}, {"prompt": "Expression Ensemble Five", "dataset": "AffectNet8", "src": "EXP-000 seed1", "UAR": 39.23, "WAR": 39.23}, {"prompt": "Expression Ensemble Five", "dataset": "FERPlus", "src": "EXP-000 seed1", "UAR": 40.65, "WAR": 51.47}, {"prompt": "Expression Ensemble Ten", "dataset": "RAFDB", "src": "EXP-000 seed1", "UAR": 57.73, "WAR": 65.91}, {"prompt": "Expression Ensemble Ten", "dataset": "AffectNet7", "src": "EXP-000 seed1", "UAR": 44.94, "WAR": 44.94}, {"prompt": "Expression Ensemble Ten", "dataset": "AffectNet8", "src": "EXP-000 seed1", "UAR": 39.55, "WAR": 39.55}, {"prompt": "Expression Ensemble Ten", "dataset": "FERPlus", "src": "EXP-000 seed1", "UAR": 40.45, "WAR": 49.29}, {"prompt": "Class Name", "dataset": "RAFDB", "src": "EXP-000 mean3", "UAR": 50.23, "WAR": 54.56}, {"prompt": "Class Name", "dataset": "AffectNet7", "src": "EXP-000 mean3", "UAR": 41.47, "WAR": 41.47}, {"prompt": "Class Name", "dataset": "AffectNet8", "src": "EXP-000 mean3", "UAR": 35.76, "WAR": 35.76}, {"prompt": "Class Name", "dataset": "FERPlus", "src": "EXP-000 mean3", "UAR": 38.58, "WAR": 45.82}, {"prompt": "An Expression of Name", "dataset": "RAFDB", "src": "EXP-000 mean3", "UAR": 55.34, "WAR": 62.0}, {"prompt": "An Expression of Name", "dataset": "AffectNet7", "src": "EXP-000 mean3", "UAR": 43.79, "WAR": 43.79}, {"prompt": "An Expression of Name", "dataset": "AffectNet8", "src": "EXP-000 mean3", "UAR": 38.4, "WAR": 38.4}, {"prompt": "An Expression of Name", "dataset": "FERPlus", "src": "EXP-000 mean3", "UAR": 38.89, "WAR": 51.62}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "RAFDB", "src": "EXP-000 mean3", "UAR": 55.64, "WAR": 64.78}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "AffectNet7", "src": "EXP-000 mean3", "UAR": 44.39, "WAR": 44.39}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "AffectNet8", "src": "EXP-000 mean3", "UAR": 38.08, "WAR": 38.08}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "FERPlus", "src": "EXP-000 mean3", "UAR": 37.95, "WAR": 49.17}, {"prompt": "Expression Ensemble Five", "dataset": "RAFDB", "src": "EXP-000 mean3", "UAR": 58.07, "WAR": 65.32}, {"prompt": "Expression Ensemble Five", "dataset": "AffectNet7", "src": "EXP-000 mean3", "UAR": 44.17, "WAR": 44.17}, {"prompt": "Expression Ensemble Five", "dataset": "AffectNet8", "src": "EXP-000 mean3", "UAR": 38.84, "WAR": 38.84}, {"prompt": "Expression Ensemble Five", "dataset": "FERPlus", "src": "EXP-000 mean3", "UAR": 41.55, "WAR": 53.41}, {"prompt": "Expression Ensemble Ten", "dataset": "RAFDB", "src": "EXP-000 mean3", "UAR": 57.19, "WAR": 64.6}, {"prompt": "Expression Ensemble Ten", "dataset": "AffectNet7", "src": "EXP-000 mean3", "UAR": 44.97, "WAR": 44.97}, {"prompt": "Expression Ensemble Ten", "dataset": "AffectNet8", "src": "EXP-000 mean3", "UAR": 39.45, "WAR": 39.45}, {"prompt": "Expression Ensemble Ten", "dataset": "FERPlus", "src": "EXP-000 mean3", "UAR": 42.17, "WAR": 49.46}, {"prompt": "Class Name", "dataset": "RAFDB", "src": "paper", "UAR": 58.7, "WAR": 65.37}, {"prompt": "Class Name", "dataset": "AffectNet7", "src": "paper", "UAR": 44.27, "WAR": 44.27}, {"prompt": "Class Name", "dataset": "AffectNet8", "src": "paper", "UAR": 38.44, "WAR": 38.43}, {"prompt": "Class Name", "dataset": "FERPlus", "src": "paper", "UAR": 48.28, "WAR": 55.42}, {"prompt": "An Expression of Name", "dataset": "RAFDB", "src": "paper", "UAR": 58.7, "WAR": 65.37}, {"prompt": "An Expression of Name", "dataset": "AffectNet7", "src": "paper", "UAR": 44.27, "WAR": 44.27}, {"prompt": "An Expression of Name", "dataset": "AffectNet8", "src": "paper", "UAR": 38.44, "WAR": 38.43}, {"prompt": "An Expression of Name", "dataset": "FERPlus", "src": "paper", "UAR": 48.28, "WAR": 55.42}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "RAFDB", "src": "paper", "UAR": 58.7, "WAR": 65.37}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "AffectNet7", "src": "paper", "UAR": 44.27, "WAR": 44.27}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "AffectNet8", "src": "paper", "UAR": 38.44, "WAR": 38.43}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "FERPlus", "src": "paper", "UAR": 48.28, "WAR": 55.42}, {"prompt": "Expression Ensemble Five", "dataset": "RAFDB", "src": "paper", "UAR": 58.7, "WAR": 65.37}, {"prompt": "Expression Ensemble Five", "dataset": "AffectNet7", "src": "paper", "UAR": 44.27, "WAR": 44.27}, {"prompt": "Expression Ensemble Five", "dataset": "AffectNet8", "src": "paper", "UAR": 38.44, "WAR": 38.43}, {"prompt": "Expression Ensemble Five", "dataset": "FERPlus", "src": "paper", "UAR": 48.28, "WAR": 55.42}, {"prompt": "Expression Ensemble Ten", "dataset": "RAFDB", "src": "paper", "UAR": 58.7, "WAR": 65.37}, {"prompt": "Expression Ensemble Ten", "dataset": "AffectNet7", "src": "paper", "UAR": 44.27, "WAR": 44.27}, {"prompt": "Expression Ensemble Ten", "dataset": "AffectNet8", "src": "paper", "UAR": 38.44, "WAR": 38.43}, {"prompt": "Expression Ensemble Ten", "dataset": "FERPlus", "src": "paper", "UAR": 48.28, "WAR": 55.42}])

def parse(path):
    rows, prompt, dataset = [], None, None
    for line in open(path):
        if "Prompt Type:" in line:
            prompt = line.split("Prompt Type:")[1].strip()
        elif re.match(r"^\*+ (RAFDB|AffectNet7|AffectNet8|FERPlus)$", line.strip()):
            dataset = line.strip().split()[-1]
        elif line.startswith("UAR/WAR:"):
            uar, war = map(float, line.split(":")[1].strip().split("/"))
            rows.append(dict(prompt=prompt, dataset=dataset, UAR=uar, WAR=war))
    return pd.DataFrame(rows)

res = pd.concat([parse(f"log/{job_of(m, mu)}-e{e}-test.txt").assign(cfg=f"{m} μ={mu:g}", epoch=e)
                 for m, mu in CONFIGS for e in SAVE_EPOCHS])
res.to_csv(f"{OUT_DIR}/exp005_sephead_l14_seed{SEED}-results.csv", index=False)
r4 = pd.read_csv(EXP004_CSV); r4 = r4[r4.lam == LAMBDA].drop(columns="lam").assign(cfg="EXP-004 λ=1 (chung)")
r3 = pd.read_csv(EXP003_CSV).drop(columns="lam", errors="ignore").assign(cfg="λ=0 (EXP-003)")
allr = pd.concat([r3[r3.epoch.isin(SAVE_EPOCHS)], r4[r4.epoch.isin(SAVE_EPOCHS)], res])
pd.set_option("display.width", 250)
DS = ["RAFDB", "AffectNet7", "AffectNet8", "FERPlus"]

ens = allr[allr.prompt == "Expression Ensemble Five"].pivot_table(index=["cfg", "epoch"], columns="dataset", values="UAR", sort=False)[DS]
b5 = REF[(REF.src == "EXP-000 seed1") & (REF.prompt == "Expression Ensemble Five")].set_index("dataset").UAR
display(pd.concat({"UAR": ens, "Δ vs EXP-000 seed1": (ens - b5).round(2)}, axis=1))

ref4 = ens.loc["EXP-004 λ=1 (chung)"]
display(pd.concat({c: (ens.loc[c] - ref4).round(2) for c in res.cfg.unique()}, names=["cfg", "epoch"]).rename_axis(columns="Δ vs EXP-004 λ=1"))

last = SAVE_EPOCHS[-1]
b = r4[r4.epoch == last].set_index(["prompt", "dataset"]).UAR
full = res[res.epoch == last].set_index(["prompt", "dataset"])
full["Δ"] = (full.UAR - b.reindex(full.index)).round(2)
display(full.reset_index().pivot_table(index=["prompt", "dataset"], columns="cfg", values="Δ", sort=False))

UAR                               Δ vs EXP-000 seed1                              
dataset                    RAFDB AffectNet7 AffectNet8 FERPlus              RAFDB AffectNet7 AffectNet8 FERPlus
cfg                 epoch                                                                                      
λ=0 (EXP-003)       5      58.64      44.20      39.02   40.56              -0.24       0.11      -0.21   -0.09
                    10     56.73      43.97      38.42   42.05              -2.15      -0.12      -0.81    1.40
                    20     54.61      44.14      37.95   45.10              -4.27       0.05      -1.28    4.45
                    30     53.68      44.20      37.85   44.96              -5.20       0.11      -1.38    4.31
EXP-004 λ=1 (chung) 5      59.39      44.09      38.70   43.27               0.51       0.00      -0.53    2.62
                    10     60.72      44.31      38.88   44.05               1.84       0.22      -0.35    3.40
                    20     59.74      45.14      39.15   45.05               0.86       1.05      -0.08    4.40
                    30     58.73      45.83      39.60   46.14              -0.15       1.74       0.37    5.49
sg μ=0              5      59.45      44.86      39.73   43.62               0.57       0.77       0.50    2.97
                    10     59.15      44.83      39.73   44.55               0.27       0.74       0.50    3.90
                    20     57.51      43.26      38.52   43.75              -1.37      -0.83      -0.71    3.10
                    30     57.38      43.03      38.35   44.33              -1.50      -1.06      -0.88    3.68
sg μ=0.1            5      60.31      45.40      40.00   43.36               1.43       1.31       0.77    2.71
                    10     60.51      45.11      40.25   44.76               1.63       1.02       1.02    4.11
                    20     59.12      45.43      39.85   46.82               0.24       1.34       0.62    6.17
                    30     57.75      45.80      40.03   47.08              -1.13       1.71       0.80    6.43
joint μ=0           5      58.17      43.71      38.98   43.86              -0.71      -0.38      -0.25    3.21
                    10     57.35      43.20      38.55   43.45              -1.53      -0.89      -0.68    2.80
                    20     55.16      42.20      37.48   43.67              -3.72      -1.89      -1.75    3.02
                    30     54.58      41.74      37.12   43.42              -4.30      -2.35      -2.11    2.77
joint μ=0.1         5      59.34      43.94      39.15   42.67               0.46      -0.15      -0.08    2.02
                    10     60.15      44.51      39.12   44.01               1.27       0.42      -0.11    3.36
                    20     59.89      45.06      38.95   46.36               1.01       0.97      -0.28    5.71
                    30     59.05      45.66      39.23   46.64               0.17       1.57       0.00    5.99

Δ vs EXP-004 λ=1   RAFDB  AffectNet7  AffectNet8  FERPlus
cfg         epoch                                        
sg μ=0      5       0.06        0.77        1.03     0.35
            10     -1.57        0.52        0.85     0.50
            20     -2.23       -1.88       -0.63    -1.30
            30     -1.35       -2.80       -1.25    -1.81
sg μ=0.1    5       0.92        1.31        1.30     0.09
            10     -0.21        0.80        1.37     0.71
            20     -0.62        0.29        0.70     1.77
            30     -0.98       -0.03        0.43     0.94
joint μ=0   5      -1.22       -0.38        0.28     0.59
            10     -3.37       -1.11       -0.33    -0.60
            20     -4.58       -2.94       -1.67    -1.38
            30     -4.15       -4.09       -2.48    -2.72
joint μ=0.1 5      -0.05       -0.15        0.45    -0.60
            10     -0.57        0.20        0.24    -0.04
            20      0.15       -0.08       -0.20     1.31
            30      0.32       -0.17       -0.37     0.50

cfg                                                      sg μ=0  sg μ=0.1  joint μ=0  joint μ=0.1
prompt                                       dataset                                             
Class Name                                   RAFDB        -4.94     -0.60      -5.17         0.37
                                             AffectNet7   -3.23      1.97      -3.54         0.69
                                             AffectNet8   -3.04      0.98      -3.27         0.33
                                             FERPlus      -0.74      3.12      -2.22         0.13
An Expression of Name                        RAFDB        -1.78      0.91      -2.13         0.38
                                             AffectNet7   -2.49      1.17      -2.80         0.00
                                             AffectNet8   -1.53      0.54      -1.68        -0.38
                                             FERPlus      -5.36      1.84      -5.22         0.33
A Photo of A Face with An Expression of Name RAFDB        -1.17     -3.19      -2.23         0.00
                                             AffectNet7   -0.31      1.31      -1.60         0.14
                                             AffectNet8    0.27      0.80      -1.03        -0.03
                                             FERPlus      -2.60     -1.83      -4.52        -2.10
Expression Ensemble Five                     RAFDB        -1.35     -0.98      -4.15         0.32
                                             AffectNet7   -2.80     -0.03      -4.09        -0.17
                                             AffectNet8   -1.25      0.43      -2.48        -0.37
                                             FERPlus      -1.81      0.94      -2.72         0.50
Expression Ensemble Ten                      RAFDB        -2.46     -1.67      -5.47         0.14
                                             AffectNet7   -2.08      0.86      -2.68         0.49
                                             AffectNet8   -0.73      1.00      -1.53         0.17
                                             FERPlus      -2.06     -0.94      -3.98        -0.09

# Log train
Accuracy train (của W_i) và loss neo trung bình ở mỗi mốc epoch. Kỳ vọng: acc ≈ λ = 0 với `sg`, loss neo ≤ EXP-004 λ = 1.

In [13]:
import re
def summary(path):
    lines = open(path).read().splitlines()
    acc = [float(l.split(":")[1]) for l in lines if l.startswith("The best accuracy")]
    anc = [float(re.search(r"Anchor [\d.]+ \(([\d.]+)\)", l).group(1)) for l in lines if "Anchor" in l]
    return {e: round(acc[e - 1], 2) for e in SAVE_EPOCHS if e <= len(acc)}, [round(a, 3) for a in anc[::max(1, len(anc) // 6)]]
ref = f"{EXP004_DIR}/exp004_anchor_crossA_l1_l14_seed1-log.txt"
if os.path.exists(ref):
    print("EXP-004 λ=1: acc %s | anchor %s" % summary(ref))
for mode, mu in CONFIGS:
    print(f"{mode} μ={mu}: acc %s | anchor %s" % summary(f"{OUT_DIR}/{job_of(mode, mu)}-log.txt"))

EXP-004 λ=1: acc {5: 39.83, 10: 60.53, 20: 75.56, 30: 81.55} | anchor [0.16, 0.085, 0.062, 0.057, 0.051, 0.051]
sg μ=0: acc {5: 41.63, 10: 61.2, 20: 75.53, 30: 81.25} | anchor [0.16, 0.258, 0.133, 0.101, 0.082, 0.073]
sg μ=0.1: acc {5: 41.63, 10: 61.2, 20: 75.53, 30: 81.25} | anchor [0.16, 0.303, 0.235, 0.242, 0.244, 0.27]
joint μ=0: acc {5: 39.77, 10: 59.93, 20: 74.94, 30: 81.0} | anchor [0.16, 0.074, 0.052, 0.044, 0.038, 0.036]
joint μ=0.1: acc {5: 39.85, 10: 60.21, 20: 74.97, 30: 80.95} | anchor [0.16, 0.092, 0.079, 0.082, 0.081, 0.088]
